# Встановила необхідні бібліотеки

In [27]:
# Базові бібліотеки для роботи з даними
import pandas as pd
import numpy as np

# Бібліотеки для завантаження та роботи з файлами
import urllib.request
import datetime
import os

# Візуалізація 
import matplotlib.pyplot as plt
import seaborn as sns

# Статистика та нормалізація
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Аналіз часу виконання 
import timeit

print("Всі інструменти готові!")

Всі інструменти готові!


In [28]:
def download_vhi_data():
    # 1. Створюємо папку, якщо її ще немає
    folder = 'vhi_data'
    if not os.path.exists(folder):
        os.makedirs(folder)
        print(f"Папка '{folder}' створена.")

    # 2. Цикл для завантаження даних для 27 областей
    for province_id in range(1, 28):
        # Формуємо URL
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        
        # Отримуємо поточну дату та час для імені файлу
        timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
        filename = f"vhi_id_{province_id}_{timestamp}.csv"
        path = os.path.join(folder, filename)

        # 3. Механізм запобігання повторного завантаження (перевірка чи є файл для цього ID)
        # Шукаємо файли, що починаються на "vhi_id_X_"
        existing_files = [f for f in os.listdir(folder) if f.startswith(f"vhi_id_{province_id}_")]
        
        if not existing_files:
            try:
                print(f"Завантажую дані для області №{province_id}...")
                urllib.request.urlretrieve(url, path)
                print(f"Збережено: {filename}")
            except Exception as e:
                print(f"Помилка при завантаженні області №{province_id}: {e}")
        else:
            print(f"Дані для області №{province_id} вже існують. Пропускаю.")

# Запуск функції
download_vhi_data()

Дані для області №1 вже існують. Пропускаю.
Дані для області №2 вже існують. Пропускаю.
Дані для області №3 вже існують. Пропускаю.
Дані для області №4 вже існують. Пропускаю.
Дані для області №5 вже існують. Пропускаю.
Дані для області №6 вже існують. Пропускаю.
Дані для області №7 вже існують. Пропускаю.
Дані для області №8 вже існують. Пропускаю.
Дані для області №9 вже існують. Пропускаю.
Дані для області №10 вже існують. Пропускаю.
Дані для області №11 вже існують. Пропускаю.
Дані для області №12 вже існують. Пропускаю.
Дані для області №13 вже існують. Пропускаю.
Дані для області №14 вже існують. Пропускаю.
Дані для області №15 вже існують. Пропускаю.
Дані для області №16 вже існують. Пропускаю.
Дані для області №17 вже існують. Пропускаю.
Дані для області №18 вже існують. Пропускаю.
Дані для області №19 вже існують. Пропускаю.
Дані для області №20 вже існують. Пропускаю.
Дані для області №21 вже існують. Пропускаю.
Дані для області №22 вже існують. Пропускаю.
Дані для області №2

# Словник відповідності ID NOAA до назв областей

In [29]:
# NOAA ID 1 - Cherkasy, 2 - Chernihiv і т.д.
noaa_id_to_name = {
    1: "Черкаська", 2: "Чернігівська", 3: "Чернівецька", 4: "Крим", 5: "Дніпропетровська",
    6: "Донецька", 7: "Івано-Франківська", 8: "Харківська", 9: "Херсонська", 10: "Хмельницька",
    11: "Київська", 12: "м. Київ", 13: "Кіровоградська", 14: "Луганська", 15: "Львівська",
    16: "Миколаївська", 17: "Одеська", 18: "Полтавська", 19: "Рівненська", 20: "Севастополь",
    21: "Сумська", 22: "Тернопільська", 23: "Закарпатська", 24: "Вінницька", 25: "Волинська",
    26: "Запорізька", 27: "Житомирська"
}

# Функція зчитування та очищення даних

In [30]:
def load_and_clean_data(folder_path):
    all_data = []
    
    # Отримуємо список усіх csv файлів у папці
    files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    
    for file in files:
        # Витягуємо ID області з назви (vhi_id_X_...)
        noaa_id = int(file.split('_')[2])
        filepath = os.path.join(folder_path, file)
        
        # Назви стовпців згідно з форматом NOAA
        headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']
        
        # видалено reg_replace. index_col=False рятує, якщо в кінці рядка є зайві коми
        df = pd.read_csv(filepath, skiprows=2, names=headers, index_col=False)
        
        # 1. Прибираємо зайвий стовпець та пропуски
        df = df.drop(columns=['empty'], errors='ignore')
        df = df.dropna()
        
        # 2. Очищення "сміття": NOAA часто ліпить <pre> прямо до року
        df['Year'] = df['Year'].astype(str).str.replace('<pre>', '', regex=False).str.replace('</pre>', '', regex=False)
        
        # 3. Фільтруємо рядки, щоб залишилися тільки цифри в роках (прибираємо можливі заголовки всередині)
        df = df[df['Year'].str.isnumeric()]
        
        # 4. Перетворюємо типи на числові для розрахунків
        df = df.astype({'Year': int, 'Week': int, 'VHI': float})
        
        # Додаємо інформацію про область
        df['area_id'] = noaa_id
        df['area_name'] = noaa_id_to_name[noaa_id]
        
        all_data.append(df)
    
    # З'єднуємо все в одну мега-таблицю
    result_df = pd.concat(all_data, ignore_index=True)
    return result_df

# Запускаємо заново
df = load_and_clean_data('vhi_data')
print("Дані готові")
df.head()

Дані готові


,Year,Week,SMN,SMT,VCI,TCI,VHI,area_id,area_name
0,1982,2,0.063,261.53,55.89,38.20,47.04,10,Хмельницька
1,1982,3,0.063,263.45,57.30,32.69,44.99,10,Хмельницька
2,1982,4,0.061,265.10,53.96,28.62,41.29,10,Хмельницька
3,1982,5,0.058,266.42,46.87,28.57,37.72,10,Хмельницька
4,1982,6,0.056,267.47,39.55,30.27,34.91,10,Хмельницька


# Завдання: Зміна індексів областей згідно з українським алфавітом

In [31]:
def reindex_vhi_data(df):
    ua_alphabet_areas = [
        "Вінницька", "Волинська", "Дніпропетровська", "Донецька", "Житомирська",
        "Закарпатська", "Запорізька", "Івано-Франківська", "Київська", "Кіровоградська",
        "Луганська", "Львівська", "Миколаївська", "Одеська", "Полтавська",
        "Рівненська", "Сумська", "Тернопільська", "Харківська", "Херсонська",
        "Хмельницька", "Черкаська", "Чернівецька", "Чернігівська", "Крим"
    ]
    
    # Створюємо мапу: Назва -> Новий ID
    reindex_map = {name: i + 1 for i, name in enumerate(ua_alphabet_areas)}
    
    # Оновлюємо area_id за допомогою мапи
    df['area_id'] = df['area_name'].map(reindex_map)
    
    # Видаляємо NaN, якщо назва області не потрапила в наш словник
    df = df.dropna(subset=['area_id'])
    df['area_id'] = df['area_id'].astype(int)
    
    return df

df = reindex_vhi_data(df)
print("Переіндексацію завершено. Тепер Вінницька область - №1.")
df[df['area_id'] == 1].head()

Переіндексацію завершено. Тепер Вінницька область - №1.


,Year,Week,SMN,SMT,VCI,TCI,VHI,area_id,area_name
33525,1982,2,0.074,265.78,67.62,23.05,45.34,1,Вінницька
33526,1982,3,0.076,267.19,69.37,20.40,44.88,1,Вінницька
33527,1982,4,0.075,268.57,65.26,17.93,41.60,1,Вінницька
33528,1982,5,0.072,269.24,58.58,20.00,39.29,1,Вінницька
33529,1982,6,0.071,270.12,52.37,22.93,37.65,1,Вінницька


# 1. Ряд VHI для області за рік 
# Це перша функція, яка витягує дані для конкретного місця і часу

In [32]:
def get_vhi_series(df, area_id, year):
    # Вибираємо рядки, де ID області та рік збігаються з нашими параметрами
    result = df[(df['area_id'] == area_id) & (df['Year'] == year)]
    # Повертаємо тільки тиждень та значення VHI
    return result[['Week', 'VHI']]

# Тестуємо на Вінницькій області (№1) за 2023 рік
print("Ряд VHI для області №1 за 2023 рік:")
get_vhi_series(df, 1, 2023)

Ряд VHI для області №1 за 2023 рік:


,Week,VHI
35656,1,40.09
35657,2,43.61
35658,3,46.74
35659,4,48.03
35660,5,47.88
35661,6,46.98
35662,7,46.11
35663,8,46.95
35664,9,48.68
35665,10,49.84


# 2. Вибірка за діапазон років та областей
# Ця функція може брати відразу кілька областей.

In [33]:
def get_vhi_range(df, area_ids, start_year, end_year):
    # .isin() дозволяє вибрати кілька ID відразу
    result = df[(df['area_id'].isin(area_ids)) & 
                (df['Year'] >= start_year) & 
                (df['Year'] <= end_year)]
    return result[['Year', 'Week', 'VHI', 'area_name']]

# Приклад: Вінницька (1) та Житомирська (5) за 2020-2021 роки
print("Вибірка для областей №1 та №5 за 2020-2021 роки:")
get_vhi_range(df, [1, 5], 2020, 2021).head(10)

Вибірка для областей №1 та №5 за 2020-2021 роки:


,Year,Week,VHI,area_name
35500,2020,1,40.92,Вінницька
35501,2020,2,43.19,Вінницька
35502,2020,3,44.74,Вінницька
35503,2020,4,45.29,Вінницька
35504,2020,5,44.80,Вінницька
35505,2020,6,43.92,Вінницька
35506,2020,7,43.10,Вінницька
35507,2020,8,42.88,Вінницька
35508,2020,9,43.71,Вінницька
35509,2020,10,45.61,Вінницька


# 3. Функція для пошуку мінімуму, максимуму та середнього

In [34]:
def get_vhi_statistics(df, area_id, year):
    # Беремо тільки стовпчик VHI для конкретної області та року
    subset = df[(df['area_id'] == area_id) & (df['Year'] == year)]['VHI']
    
    return {
        "Min VHI": subset.min(),
        "Max VHI": subset.max(),
        "Average": subset.mean(),
        "Median": subset.median()
    }

# Виведемо статистику для області №1 за 2022 рік
print(f"Статистика за 2022 рік: {get_vhi_statistics(df, 1, 2022)}")

Статистика за 2022 рік: {'Min VHI': np.float64(32.28), 'Max VHI': np.float64(56.9), 'Average': np.float64(43.281346153846165), 'Median': np.float64(42.53)}
